# Notebook Overview — Build Dataset (Optional)

## Purpose

This notebook provides an **optional path** for building a raw image dataset from **one selected source at a time**.

It downloads candidate images from the selected source, applies basic validation and deduplication, assigns standardized filenames, and creates raw metadata records for accepted images.

The notebook also supports a **demo mode** in which the dataset is treated as if it does not already exist, while no images, CSV files, or hash files are written to disk.

---

## Inputs

The notebook supports the following dataset sources:

* ImageNet_1K_256
* MS_COCO_2017
* DiffusionDB
* SDXL_Generated_10K
* MidJourney
* OpenImages

**OpenImages is not supported in this notebook** due to its very large size.

The selected source is controlled through the interactive dataset selection cell.

---

## Assumptions

* This notebook is **optional**
* The user may choose to:
  * download one dataset source, or
  * skip Notebook 01 entirely
* Only **one dataset source** is processed per run
* Target-specific logic is implemented in `src/targets/*.py`
* Accepted images must satisfy:
  * **width ≥ 256**
  * **height ≥ 256**
* Duplicate detection is performed using **SHA-256 hashes**
* Accepted images are assigned standardized filenames
* Raw metadata records are created for accepted images
* No preprocessing is performed here
* No dataset splitting is performed here

---

## Demo Mode

The notebook supports a configurable execution mode using:

`PERSIST_OUTPUTS = False`

In this mode:

* existing files are ignored for the current run
* the selected dataset is treated as if it has not yet been built
* images are **not** written to disk
* metadata CSV files are **not** written to disk
* hash JSON files are **not** written to disk
* metadata rows and hash sets are maintained **in memory only**

This mode is useful for:
* testing notebook behavior
* demonstrating dataset construction
* avoiding overwriting or reusing existing source data

---

## Dataset Categories

| Target Name  | Dataset            | Class Label | Dataset Code |
|--------------|--------------------|-------------|--------------|
| imagenet     | ImageNet_1K_256    | rl          | imgn         |
| coco         | MS_COCO_2017       | rl          | coco         |
| openimages   | OpenImages         | rl          | open
| diffusiondb  | DiffusionDB        | ai          | diff         |
| sdxl         | SDXL_Generated_10K | ai          | sdxl         |
| midjourney   | MidJourney         | ai          | mj           |

---

## What This Notebook Does

### Cell 1 — Startup / Environment Setup

* Mount Google Drive
* Configure project paths
* Import shared configuration
* Define `RAW_DIR`
* Set demo or persistent mode with `PERSIST_OUTPUTS`

---

### Cell 2 — Download Mode and Target Selection

* Allow the user to:
  * download one dataset source, or
  * skip Notebook 01
* Block unsupported OpenImages selection

---

### Cell 3 — Load Target Module

* Dynamically import the selected target module
* Resolve dataset-specific display name, class label, and dataset code

---

### Cell 4 — Common Configuration

* Define dataset-specific paths for:
  * raw image directory
  * metadata CSV
  * source hash file
  * global hash file

---

### Cell 5 — Raw Metadata CSV Setup

* Define raw metadata columns
* In persistent mode:
  * initialize raw metadata CSV
* In demo mode:
  * store metadata rows in memory only

---

### Cell 6 — Hash Utilities

* Load and save SHA-256 hash sets
* In persistent mode:
  * reuse or create source/global hash files
* In demo mode:
  * use in-memory hash sets only

---

### Cell 7 — Filename and Counting Utilities

* Count accepted images
* Determine next image index
* Generate standardized filenames

---

### Cell 8 — Image Utilities

* Normalize images to RGB
* Check minimum size requirement
* Compute SHA-256 hash from normalized PNG content

---

### Cell 9 — CSV Append Utility

* Append accepted metadata rows
* In persistent mode:
  * write rows to CSV
* In demo mode:
  * store rows in memory

---

### Cell 10 — Load Source Dataset

* Load the selected source dataset through the target module
* Initialize starting state for the build process

---

### Cell 11 — Preview Candidate Records

* Retrieve a small preview batch
* Display source identifiers and basic image information

---

### Cell 12 — Process One Candidate Record

* Normalize image format
* Apply minimum size filtering
* Compute SHA-256 hash
* Reject duplicates using source and global hash sets
* Assign filename and saved path
* Save accepted images only in persistent mode
* Append metadata row for each accepted image

---

### Cell 13 — Build Dataset for Selected Source

* Iterate through candidate batches
* Handle both:
  * target modules with `start_index` support
  * iterable datasets using notebook-side iteration
* Continue until target accepted count is reached or records are exhausted
* Display build progress and batch statistics

---

### Cell 14 — Verify Saved Output

* In persistent mode:
  * compare saved image count and metadata CSV row count
* In demo mode:
  * compare in-memory metadata row count and hash counts

---

### Cell 15 — Run Dataset Build and Verification

* Execute the build loop
* Print summary statistics
* Run verification

---

### Cell 16 — Final Summary Report

* Print final dataset summary
* Show counts and acceptance rate
* Display sample metadata rows in demo mode

---

## Outputs

### Persistent Mode (`PERSIST_OUTPUTS = True`)

For the selected dataset source, the notebook generates:

* saved PNG image files in:

  * `data/raw/<SOURCE_DATASET>/images/`

* raw metadata CSV:

  * `data/metadata/[dataset_code]_raw_metadata.csv`

* per-source hash file:

  * `data/metadata/[target_name]_source_hashes.json`

It also updates:

* `data/metadata/global_hashes.json`

### Demo Mode (`PERSIST_OUTPUTS = False`)

* no files are written
* accepted image metadata rows are stored in memory
* source and global hash sets are stored in memory

---

## Expected Sizes

* Target accepted images per source: approximately **3000**
* Full project total across all five supported sources in Notebook 01: **15,000**
* Full project total across all six project sources: **18,000**

---

## Notes

* This notebook is optional
* It supports building **one source at a time**
* OpenImages is intentionally blocked in Notebook 01
* Duplicate detection uses both:
  * per-source hash tracking
  * global hash tracking
* Demo mode is the recommended setting when existing source data should be ignored
* This notebook performs raw dataset construction and metadata generation only
* No preprocessing or dataset splitting is performed here


In [ ]:
# ============================================
# Cell 1: Startup / Environment Setup
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
from pathlib import Path

# -------------------------------------------------
# Project paths
# -------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"
PROJECT_DIR = os.path.join(BASE_DRIVE_DIR, "src")

# -------------------------------------------------
# Add project src directory to Python path
# -------------------------------------------------
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# -------------------------------------------------
# Import shared config
# -------------------------------------------------
from project_config import (
    BASE_DIR,
    DATA_DIR,
    METADATA_DIR,
    AI_LABEL,
    REAL_LABEL,
)

# -------------------------------------------------
# Define RAW_DIR locally
# -------------------------------------------------
RAW_DIR = os.path.join(DATA_DIR, "raw")

# -------------------------------------------------
# Demo / persistence mode
# False = act like dataset does not exist yet, save nothing
# True  = normal persistent behavior
# -------------------------------------------------
PERSIST_OUTPUTS = False

# -------------------------------------------------
# Ensure required directories exist
# -------------------------------------------------
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

print("Environment setup complete.\n")
print(f"BASE_DIR        : {BASE_DIR}")
print(f"DATA_DIR        : {DATA_DIR}")
print(f"METADATA_DIR    : {METADATA_DIR}")
print(f"RAW_DIR         : {RAW_DIR}")
print(f"PERSIST_OUTPUTS : {PERSIST_OUTPUTS}")

if PERSIST_OUTPUTS:
    print("\nPersistent mode enabled.")
    print("Images, metadata, and hash files will be written.")
else:
    print("\nDemo mode enabled.")
    print("Existing files will be ignored for this run.")
    print("No images, metadata CSVs, or hash JSON files will be written.")



In [ ]:
# ============================================
# Cell 2: Download Mode and Target Selection
# ============================================

import ipywidgets as widgets
from IPython.display import display, clear_output

RUN_DOWNLOAD = False
TARGET_NAME = None

mode_widget = widgets.Dropdown(
    options=[
        ("Download one dataset", "download"),
        ("Skip Notebook 01", "skip"),
    ],
    value="download",
    description="Mode:"
)

dataset_widget = widgets.Dropdown(
    options=[
        ("ImageNet (SMALL ~331 MB)", "imagenet"),
        ("MS COCO 2017 (MEDIUM ~1.5 GB)", "coco"),
        ("DiffusionDB (MEDIUM ~1.8 GB)", "diffusiondb"),
        ("SDXL (VERY LARGE ~3.7 GB)", "sdxl"),
        ("MidJourney (VERY LARGE ~3.8 GB)", "midjourney"),
        ("OpenImages (EXTREME >20 GB — NOT SUPPORTED)", "openimages"),
    ],
    value="imagenet",
    description="Dataset:"
)

apply_button = widgets.Button(
    description="Apply Selection",
    button_style="primary"
)

output = widgets.Output()

def on_apply_clicked(b):
    global RUN_DOWNLOAD, TARGET_NAME

    with output:
        clear_output()

        if mode_widget.value == "skip":
            RUN_DOWNLOAD = False
            TARGET_NAME = None
            print("Notebook 01 skipped.")
            print("Proceed to 02_Preprocess_Images.ipynb.")
            return

        RUN_DOWNLOAD = True
        TARGET_NAME = dataset_widget.value

        if TARGET_NAME == "openimages":
            RUN_DOWNLOAD = False
            TARGET_NAME = None
            print("OpenImages is too large (>20 GB).")
            print("This dataset is not supported in Notebook 01.")
            print("Please select a different dataset or skip Notebook 01.")
            return

        size_map = {
            "imagenet": "SMALL (~331 MB)",
            "coco": "MEDIUM (~1.5 GB)",
            "diffusiondb": "MEDIUM (~1.8 GB)",
            "sdxl": "VERY LARGE (~3.7 GB)",
            "midjourney": "VERY LARGE (~3.8 GB)",
        }

        print("Selection Summary:")
        print(f"RUN_DOWNLOAD : {RUN_DOWNLOAD}")
        print(f"TARGET_NAME  : {TARGET_NAME}")
        print(f"Dataset Size : {size_map[TARGET_NAME]}")

apply_button.on_click(on_apply_clicked)

display(mode_widget)
display(dataset_widget)
display(apply_button)
display(output)



In [ ]:
# ============================================
# Cell 3: Load Target Module
# ============================================

import importlib

TARGET_SPECS = {
    "diffusiondb": {
        "display_name": "DiffusionDB",
        "module_name": "diffusiondb_target",
        "source_dataset": "DiffusionDB",
        "class_label": "ai",
        "dataset_code": "diff",
    },
    "sdxl": {
        "display_name": "SDXL Generated (10K)",
        "module_name": "sdxl_target",
        "source_dataset": "SDXL_Generated_10K",
        "class_label": "ai",
        "dataset_code": "sdxl",
    },
    "imagenet": {
        "display_name": "ImageNet (256)",
        "module_name": "imagenet_target",
        "source_dataset": "ImageNet_1K_256",
        "class_label": "rl",
        "dataset_code": "imgn",
    },
    "coco": {
        "display_name": "MS COCO 2017",
        "module_name": "coco_target",
        "source_dataset": "MS_COCO_2017",
        "class_label": "rl",
        "dataset_code": "coco",
    },
    "midjourney": {
        "display_name": "Midjourney",
        "module_name": "midjourney_target",
        "source_dataset": "Midjourney",
        "class_label": "ai",
        "dataset_code": "mj",
    },
    "openimages": {
        "display_name": "OpenImages",
        "module_name": "openimages_target",
        "source_dataset": "OpenImages",
        "class_label": "rl",
        "dataset_code": "open",
    },
}

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping module load (no valid dataset download selected).")

    TARGET_SPEC = None
    DISPLAY_NAME = None
    MODULE_NAME = None
    SOURCE_DATASET = None
    CLASS_LABEL = None
    DATASET_CODE = None
    target_module = None

else:
    if TARGET_NAME not in TARGET_SPECS:
        raise ValueError(f"Unsupported TARGET_NAME: {TARGET_NAME}")

    TARGET_SPEC = TARGET_SPECS[TARGET_NAME]

    DISPLAY_NAME   = TARGET_SPEC["display_name"]
    MODULE_NAME    = TARGET_SPEC["module_name"]
    SOURCE_DATASET = TARGET_SPEC["source_dataset"]
    CLASS_LABEL    = TARGET_SPEC["class_label"]
    DATASET_CODE   = TARGET_SPEC["dataset_code"]

    target_module = importlib.import_module(f"targets.{MODULE_NAME}")
    importlib.reload(target_module)

    print("Target module loaded successfully.\n")
    print(f"DISPLAY_NAME   : {DISPLAY_NAME}")
    print(f"MODULE_NAME    : {MODULE_NAME}")
    print(f"SOURCE_DATASET : {SOURCE_DATASET}")
    print(f"CLASS_LABEL    : {CLASS_LABEL}")
    print(f"DATASET_CODE   : {DATASET_CODE}")



In [ ]:
# ============================================
# Cell 4: Common Configuration
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping common configuration (no valid dataset download selected).")

    DATASET_RAW_DIR = None
    IMAGE_OUTPUT_DIR = None
    RAW_METADATA_CSV = None
    SOURCE_HASHES_JSON = None
    GLOBAL_HASHES_JSON = None

else:
    DATASET_RAW_DIR = os.path.join(RAW_DIR, SOURCE_DATASET)
    IMAGE_OUTPUT_DIR = os.path.join(DATASET_RAW_DIR, "images")

    RAW_METADATA_CSV = os.path.join(
        METADATA_DIR,
        f"{DATASET_CODE}_raw_metadata.csv"
    )

    SOURCE_HASHES_JSON = os.path.join(
        METADATA_DIR,
        f"{TARGET_NAME}_source_hashes.json"
    )

    GLOBAL_HASHES_JSON = os.path.join(
        METADATA_DIR,
        "global_hashes.json"
    )

    os.makedirs(DATASET_RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
    os.makedirs(METADATA_DIR, exist_ok=True)

    print("Common configuration complete.\n")
    print(f"DATASET_RAW_DIR    : {DATASET_RAW_DIR}")
    print(f"IMAGE_OUTPUT_DIR   : {IMAGE_OUTPUT_DIR}")
    print(f"RAW_METADATA_CSV   : {RAW_METADATA_CSV}")
    print(f"SOURCE_HASHES_JSON : {SOURCE_HASHES_JSON}")
    print(f"GLOBAL_HASHES_JSON : {GLOBAL_HASHES_JSON}")



In [ ]:
# ============================================
# Cell 5: Raw Metadata CSV Setup
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping raw metadata CSV setup (no valid dataset download selected).")
    RAW_METADATA_COLUMNS = None
    demo_metadata_rows = []

else:
    RAW_METADATA_COLUMNS = [
        "filename",
        "label",
        "dataset_code",
        "source_name",
        "source_id",
        "source_ref",
        "original_width",
        "original_height",
        "saved_path",
        "sha256",
        "batch_id",
    ]

    demo_metadata_rows = []

    if PERSIST_OUTPUTS:
        if os.path.exists(RAW_METADATA_CSV):
            print("WARNING: Raw metadata CSV already exists and will be overwritten:")
            print(f"  {RAW_METADATA_CSV}")
        else:
            print("Creating new raw metadata CSV:")
            print(f"  {RAW_METADATA_CSV}")

        with open(RAW_METADATA_CSV, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(RAW_METADATA_COLUMNS)

        print("\nRaw metadata CSV initialized.")

    else:
        print("Demo mode enabled.")
        print("Metadata rows will be created in memory only.")
        print("No raw metadata CSV file will be written.")

    print("\nMetadata columns:")
    for col in RAW_METADATA_COLUMNS:
        print(f"  - {col}")



In [ ]:
# ============================================
# Cell 6: Hash Utilities
# ============================================

import json

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping hash utilities setup (no valid dataset download selected).")

    source_hashes = set()
    global_hashes = set()

    def load_hash_set(json_path):
        return set()

    def save_hash_set(hash_set, json_path):
        return

else:
    def load_hash_set(json_path):
        if os.path.exists(json_path):
            with open(json_path, "r") as f:
                return set(json.load(f))
        return set()

    def save_hash_set(hash_set, json_path):
        if not PERSIST_OUTPUTS:
            return
        with open(json_path, "w") as f:
            json.dump(sorted(hash_set), f, indent=2)

    if PERSIST_OUTPUTS:
        source_hashes = load_hash_set(SOURCE_HASHES_JSON)
        global_hashes = load_hash_set(GLOBAL_HASHES_JSON)

        if os.path.exists(SOURCE_HASHES_JSON):
            print("Source hash file already exists and will be reused:")
            print(f"  {SOURCE_HASHES_JSON}")
        else:
            print("Source hash file will be created:")
            print(f"  {SOURCE_HASHES_JSON}")

        if os.path.exists(GLOBAL_HASHES_JSON):
            print("Global hash file already exists and will be reused:")
            print(f"  {GLOBAL_HASHES_JSON}")
        else:
            print("Global hash file will be created:")
            print(f"  {GLOBAL_HASHES_JSON}")

        print("\nLoaded hash counts:")
        print(f"  source_hashes : {len(source_hashes)}")
        print(f"  global_hashes : {len(global_hashes)}")

    else:
        source_hashes = set()
        global_hashes = set()

        print("Demo mode enabled.")
        print("Starting with empty in-memory hash sets.")
        print("No hash files will be read or written.")

        print("\nLoaded hash counts:")
        print(f"  source_hashes : {len(source_hashes)}")
        print(f"  global_hashes : {len(global_hashes)}")



In [ ]:
# ============================================
# Cell 7: Filename and Counting Utilities
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping filename and counting utilities (no valid dataset download selected).")

    def count_saved_images():
        return 0

    def get_next_image_index():
        return 1

    def make_filename(index):
        return None

else:
    def count_saved_images():
        if not PERSIST_OUTPUTS:
            return 0
        if not os.path.exists(IMAGE_OUTPUT_DIR):
            return 0
        return len([
            f for f in os.listdir(IMAGE_OUTPUT_DIR)
            if f.lower().endswith(".png")
        ])

    def get_next_image_index():
        return count_saved_images() + 1

    def make_filename(index):
        return f"{DATASET_CODE}_{index:06d}.png"

    current_count = count_saved_images()
    next_index = get_next_image_index()
    sample_filename = make_filename(next_index)

    if PERSIST_OUTPUTS:
        print("Persistent mode enabled.")
    else:
        print("Demo mode enabled. Existing saved images will be ignored.")

    print(f"\nCurrent saved image count : {current_count}")
    print(f"Next image index          : {next_index}")
    print(f"Next filename example     : {sample_filename}")



In [ ]:
# ============================================
# Cell 8: Image Utilities
# ============================================

import io
import hashlib
from PIL import Image

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping image utilities (no valid dataset download selected).")

    def normalize_to_rgb(image_obj):
        return None

    def meets_min_size(image_obj, min_size=256):
        return False

    def compute_image_sha256(image_obj):
        return None

else:
    def normalize_to_rgb(image_obj):
        """
        Convert input image to RGB PIL Image.
        """
        if image_obj.mode != "RGB":
            image_obj = image_obj.convert("RGB")
        return image_obj

    def meets_min_size(image_obj, min_size=256):
        """
        Return True if both width and height meet minimum size.
        """
        width, height = image_obj.size
        return width >= min_size and height >= min_size

    def compute_image_sha256(image_obj):
        """
        Compute SHA-256 hash from normalized PNG byte content.
        """
        image_obj = normalize_to_rgb(image_obj)

        buffer = io.BytesIO()
        image_obj.save(buffer, format="PNG")
        image_bytes = buffer.getvalue()

        return hashlib.sha256(image_bytes).hexdigest()

    print("Image utilities ready.\n")

    # Quick self-check using a synthetic image
    test_img = Image.new("RGB", (256, 256), color=(0, 0, 0))
    test_hash = compute_image_sha256(test_img)

    print(f"Minimum size check passed : {meets_min_size(test_img)}")
    print(f"Example SHA-256 length    : {len(test_hash)}")
    print(f"Example SHA-256 prefix    : {test_hash[:16]}...")



In [ ]:
# ============================================
# Cell 9: CSV Append Utility
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping CSV append utility (no valid dataset download selected).")

    demo_metadata_rows = []

    def append_metadata_row(row_dict):
        return

else:
    demo_metadata_rows = []

    def append_metadata_row(row_dict):
        """
        Append one accepted image metadata row.
        Writes to CSV in persistent mode, or stores in memory in demo mode.
        """
        if PERSIST_OUTPUTS:
            with open(RAW_METADATA_CSV, "a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=RAW_METADATA_COLUMNS)
                writer.writerow(row_dict)
        else:
            demo_metadata_rows.append(row_dict)

    if PERSIST_OUTPUTS:
        print("CSV append utility ready.")
        print(f"Target CSV : {RAW_METADATA_CSV}")
    else:
        print("Demo mode enabled.")
        print("Metadata rows will be stored in memory only.")


In [ ]:
# ============================================
# Cell 10: Load Source Dataset
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping source dataset load (no valid dataset download selected).")
    dataset_obj = None

else:
    print("Loading source dataset...\n")

    try:
        dataset_obj = target_module.load_source_dataset()
    except Exception as e:
        print("ERROR: Failed to load source dataset.")
        raise e

    print("Source dataset loaded successfully.\n")
    print(f"Dataset type : {type(dataset_obj)}")

    # Show starting state (consistent with demo mode)
    if PERSIST_OUTPUTS:
        current_count = count_saved_images()
        print(f"\nCurrent accepted images (from disk) : {current_count}")
    else:
        print("\nDemo mode: starting from empty in-memory state.")
        print("Accepted image count : 0")



In [ ]:
# ============================================
# Cell 11: Preview Candidate Records
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate preview (no valid dataset download selected).")

else:
    print("Retrieving candidate records...\n")

    try:
        # Get a small preview batch
        preview_records = target_module.get_candidate_records(dataset_obj, batch_size=5)
    except Exception as e:
        print("ERROR: Failed to retrieve candidate records.")
        raise e

    if not preview_records:
        print("No candidate records returned.")
    else:
        print(f"Retrieved {len(preview_records)} candidate records.\n")

        for i, record in enumerate(preview_records, start=1):
            print(f"Record {i}:")
            print(f"  source_id  : {record.get('source_id')}")
            print(f"  source_ref : {record.get('source_ref')}")

            img = record.get("image_obj")
            if img is not None:
                try:
                    print(f"  image size : {img.size}")
                    print(f"  image mode : {img.mode}")
                except Exception:
                    print("  image info : <unavailable>")
            else:
                print("  image_obj  : None")

            print()



In [ ]:
# ============================================
# Cell 12: Process One Candidate Record
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping candidate record processing (no valid dataset download selected).")

    def process_candidate_record(record, index):
        return False

else:
    def process_candidate_record(record, index):
        """
        Process a single candidate record.
        Returns True if accepted, False otherwise.
        """

        try:
            source_id = record.get("source_id")
            source_ref = record.get("source_ref")
            image_obj = record.get("image_obj")

            if image_obj is None:
                return False

            # Normalize
            image_obj = normalize_to_rgb(image_obj)

            # Size filter
            if not meets_min_size(image_obj):
                return False

            # Compute hash
            sha256 = compute_image_sha256(image_obj)

            # Deduplication
            if sha256 in source_hashes:
                return False

            if sha256 in global_hashes:
                return False

            # Generate filename
            filename = make_filename(index)

            # Save path (real or demo)
            if PERSIST_OUTPUTS:
                saved_path = os.path.join(IMAGE_OUTPUT_DIR, filename)
                image_obj.save(saved_path, format="PNG")
            else:
                saved_path = f"[demo_only]/{filename}"

            # Update hash sets
            source_hashes.add(sha256)
            global_hashes.add(sha256)

            # Build metadata row
            row = {
                "filename": filename,
                "label": CLASS_LABEL,
                "dataset_code": DATASET_CODE,
                "source_name": SOURCE_DATASET,
                "source_id": source_id,
                "source_ref": source_ref,
                "original_width": image_obj.size[0],
                "original_height": image_obj.size[1],
                "saved_path": saved_path,
                "sha256": sha256,
                "batch_id": None,
            }

            append_metadata_row(row)

            return True

        except Exception:
            return False

    print("Candidate record processing function ready.")



In [ ]:
# ============================================
# Cell 13: Build Dataset for Selected Source
# ============================================

from tqdm.notebook import tqdm
import inspect
import itertools

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build (no valid dataset download selected).")

else:
    def build_dataset_for_selected_source(target_count=3000, batch_size=100, max_batches=None):
        """
        Build dataset for the selected source until target_count accepted images are reached
        or candidate records are exhausted.
        Supports:
          - target modules with start_index support
          - iterable datasets via notebook-side iteration
        """

        accepted_count = 0
        rejected_count = 0
        processed_count = 0
        batch_id = 0
        next_index = get_next_image_index()

        print("Starting dataset build...\n")
        print(f"Target source     : {SOURCE_DATASET}")
        print(f"Target count      : {target_count}")
        print(f"Batch size        : {batch_size}")
        print(f"Persistence mode  : {PERSIST_OUTPUTS}\n")

        sig = inspect.signature(target_module.get_candidate_records)
        supports_start_index = "start_index" in sig.parameters

        if supports_start_index:
            batching_mode = "module start_index"
            source_iter = None
            candidate_offset = 0
        else:
            batching_mode = "notebook iterator"
            source_iter = iter(dataset_obj)
            candidate_offset = None

        print(f"Batching mode     : {batching_mode}\n")

        pbar = tqdm(total=target_count, desc="Accepted Images", unit="img")

        while accepted_count < target_count:
            batch_id += 1

            if max_batches is not None and batch_id > max_batches:
                print("Reached max_batches limit.")
                break

            try:
                if supports_start_index:
                    candidate_records = target_module.get_candidate_records(
                        dataset_obj,
                        start_index=candidate_offset,
                        batch_size=batch_size
                    )
                    candidate_offset += batch_size
                else:
                    raw_batch = list(itertools.islice(source_iter, batch_size))
                    if not raw_batch:
                        candidate_records = []
                    else:
                        candidate_records = target_module.get_candidate_records(
                            raw_batch,
                            batch_size=batch_size
                        )

            except Exception as e:
                print(f"ERROR: Failed to retrieve candidate records for batch {batch_id}.")
                raise e

            if not candidate_records:
                print("No more candidate records returned. Stopping build.")
                break

            batch_accepted = 0
            batch_rejected = 0

            for record in candidate_records:
                processed_count += 1

                accepted = process_candidate_record(record, next_index)

                if accepted:
                    accepted_count += 1
                    batch_accepted += 1
                    next_index += 1
                    pbar.update(1)
                else:
                    rejected_count += 1
                    batch_rejected += 1

                if accepted_count >= target_count:
                    break

            if PERSIST_OUTPUTS:
                save_hash_set(source_hashes, SOURCE_HASHES_JSON)
                save_hash_set(global_hashes, GLOBAL_HASHES_JSON)

            if supports_start_index:
                offset_text = f"{candidate_offset:5d}"
            else:
                offset_text = "iter "

            print(
                f"Batch {batch_id:03d} | "
                f"offset: {offset_text} | "
                f"processed: {len(candidate_records):4d} | "
                f"accepted: {batch_accepted:4d} | "
                f"rejected: {batch_rejected:4d} | "
                f"total accepted: {accepted_count:4d}"
            )

            if len(candidate_records) < batch_size:
                print("Final partial batch reached. Stopping build.")
                break

        pbar.close()

        print("\nDataset build complete.\n")
        print(f"Total processed : {processed_count}")
        print(f"Total accepted  : {accepted_count}")
        print(f"Total rejected  : {rejected_count}")

        if PERSIST_OUTPUTS:
            print(f"Images saved to : {IMAGE_OUTPUT_DIR}")
            print(f"Metadata CSV    : {RAW_METADATA_CSV}")
        else:
            print("Demo mode: no files were written.")
            print(f"In-memory metadata rows : {len(demo_metadata_rows)}")

        return {
            "processed_count": processed_count,
            "accepted_count": accepted_count,
            "rejected_count": rejected_count,
            "batches_run": batch_id,
        }

    print("Dataset build function ready.")



In [ ]:
# ============================================
# Cell 14: Verify Saved Output
# ============================================

import csv

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping output verification (no valid dataset download selected).")

else:
    def verify_saved_output():
        print("Verifying output...\n")

        if PERSIST_OUTPUTS:
            # ------------------------------------
            # Persistent mode verification
            # ------------------------------------
            image_count = count_saved_images()

            if os.path.exists(RAW_METADATA_CSV):
                with open(RAW_METADATA_CSV, "r", newline="") as f:
                    reader = csv.reader(f)
                    row_count = sum(1 for _ in reader) - 1  # exclude header
            else:
                row_count = 0

            print(f"Saved images       : {image_count}")
            print(f"Metadata CSV rows  : {row_count}")

            if image_count == row_count:
                print("\nVerification passed: image count matches metadata row count.")
            else:
                print("\nWARNING: image count does not match metadata row count.")

            return {
                "image_count": image_count,
                "metadata_row_count": row_count,
                "match": (image_count == row_count),
            }

        else:
            # ------------------------------------
            # Demo mode verification
            # ------------------------------------
            metadata_row_count = len(demo_metadata_rows)
            source_hash_count = len(source_hashes)
            global_hash_count = len(global_hashes)

            print(f"In-memory metadata rows : {metadata_row_count}")
            print(f"Source hash count       : {source_hash_count}")
            print(f"Global hash count       : {global_hash_count}")

            rows_match_source_hashes = (metadata_row_count == source_hash_count)

            if rows_match_source_hashes:
                print("\nVerification passed: metadata row count matches source hash count.")
            else:
                print("\nWARNING: metadata row count does not match source hash count.")

            return {
                "metadata_row_count": metadata_row_count,
                "source_hash_count": source_hash_count,
                "global_hash_count": global_hash_count,
                "match": rows_match_source_hashes,
            }

    print("Output verification function ready.")



In [ ]:
# ============================================
# Cell 15: Run Dataset Build and Verification
# ============================================

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("Skipping dataset build and verification (no valid dataset selected).")

else:
    print("Running dataset build...\n")

    # -------------------------------------------------
    # Run builder (you can adjust target_count if needed)
    # -------------------------------------------------
    results = build_dataset_for_selected_source(
        target_count=3000,
        batch_size=100
    )

    print("\nBuild Summary:")
    print(f"Processed : {results['processed_count']}")
    print(f"Accepted  : {results['accepted_count']}")
    print(f"Rejected  : {results['rejected_count']}")
    print(f"Batches   : {results['batches_run']}")

    print("\nRunning verification...\n")

    verification = verify_saved_output()

    print("\nVerification Summary:")
    for key, value in verification.items():
        print(f"{key} : {value}")

    print("\nNotebook 01 complete.")



In [ ]:
# ============================================
# Cell 16: Final Summary Report
# ============================================

import time

if not RUN_DOWNLOAD or TARGET_NAME is None:
    print("No dataset was built in this session.")

else:
    print("========== FINAL SUMMARY ==========\n")

    print(f"Source Dataset : {SOURCE_DATASET}")
    print(f"Dataset Code   : {DATASET_CODE}")
    print(f"Persistence    : {PERSIST_OUTPUTS}")

    # ----------------------------------------
    # Counts
    # ----------------------------------------
    if PERSIST_OUTPUTS:
        image_count = count_saved_images()

        print(f"\nSaved Images   : {image_count}")
        print(f"Metadata CSV   : {RAW_METADATA_CSV}")
        print(f"Image Dir      : {IMAGE_OUTPUT_DIR}")

    else:
        image_count = len(demo_metadata_rows)
        print(f"\nAccepted Images (in-memory) : {image_count}")

    print(f"Source Hashes  : {len(source_hashes)}")
    print(f"Global Hashes  : {len(global_hashes)}")

    # ----------------------------------------
    # Efficiency metrics (if results available)
    # ----------------------------------------
    try:
        processed = results["processed_count"]
        accepted  = results["accepted_count"]

        accept_rate = accepted / processed if processed > 0 else 0

        print("\n--- Performance ---")
        print(f"Total Processed : {processed}")
        print(f"Total Accepted  : {accepted}")
        print(f"Acceptance Rate : {accept_rate:.4f}")

    except Exception:
        print("\n(No performance stats available)")

    # ----------------------------------------
    # Sample metadata preview (demo mode)
    # ----------------------------------------
    if not PERSIST_OUTPUTS and len(demo_metadata_rows) > 0:
        print("\n--- Sample Metadata Rows ---\n")

        preview_count = min(3, len(demo_metadata_rows))

        for i in range(preview_count):
            print(f"Row {i+1}:")
            for k, v in demo_metadata_rows[i].items():
                print(f"  {k}: {v}")
            print()

    # ----------------------------------------
    # Completion message
    # ----------------------------------------
    print("===================================")
    print("\nNotebook 01 execution complete.")

